# Why use OpenMS

OpenMS is an open-source, C++ framework hosted on Github <https://github.com/OpenMS/OpenMS> for analyzing large volumes of mass spectrometry data. It has been specially designed for analyzing high performance LC-MS data but over recent times, has been extended to analyze data generated by other techniques.

https://pyopenms.readthedocs.io/en/latest/user_guide/background.html


Installing these programmes by uv is much faster than installing via pip. Here is why uv excels with this specific stack:

1. **Massive Parallel Downloads & Compilations:** Libraries like rdkit, matchms, ms2deepscore, and their dependencies (numba, scipy, matplotlib, torch) are heavy. Standard pip downloads and builds them sequentially, whereas uv processes downloads, metadata extraction, and wheel building in parallel.

2. **Rust-Powered Engine:** uv is written in Rust and operates directly at the system level. It bypasses Python's interpreter overhead for resolving dependency graphs.

3. **Immediate Mismatch Resolution:** In your notebook's install log, pip spent significant time downloading and building packages before finally failing late with: ERROR: Package 'neims-pytorch' requires a different Python: 3.13.16 not in '==3.12.*'. uv resolves the entire dependency tree before downloading any binaries. It would have detected this environment conflict within milliseconds, saving you minutes of wasted download time.

In [ ]:
#@title Install required Python libraries
!uv pip install --system -q ipyketcher rdkit mols2grid pyteomics pyopenms psims lxml hdf5plugin pandas pubchempy mendeleev marimo wigglystuff matchms onnxruntime bottleneck

import os
import glob
import pandas as pd

# 1. Clone hplc-py for chromatographic deconvolution
if not os.path.exists('hplc-py'):
    !git clone -q https://github.com/cremerlab/hplc-py.git

In [ ]:
#@title Plot HPLC Chromatogram, Convert the HPLC Trace into a Pandas dataframe
import urllib.request
import os
import matplotlib.pyplot as plt
from pyteomics import mzml
import pandas as pd

# 1. Download the new dataset
url_new = "https://raw.githubusercontent.com/OpenMS/OpenMS/develop/src/tests/topp/FeatureFinderMetaboIdent_1_input.mzML"
filename_new = "FeatureFinderMetaboIdent_1_input.mzML"

if not os.path.exists(filename_new):
    print(f"Downloading {filename_new}...")
    urllib.request.urlretrieve(url_new, filename_new)

# 2. Parse the new dataset
print(f"Parsing {filename_new}...")
rt_new = []
tic_new = []

try:
    with mzml.read(filename_new) as reader:
        for spectrum in reader:
            if spectrum['ms level'] == 1:
                scan = spectrum.get('scanList', {}).get('scan', [{}])[0]
                rt = scan.get('scan start time', 0)
                tic = spectrum.get('total ion current', 0)
                rt_new.append(rt)
                tic_new.append(tic)

    # Create a pandas DataFrame from the parsed data
    hplc_df = pd.DataFrame({
        'Signal': tic_new,
        'Time': rt_new
    })

    print("HPLC Trace DataFrame successfully created:")
    print(hplc_df.head())

    # 3. Plot the new HPLC trace using the DataFrame
    plt.figure(figsize=(10, 6))
    plt.plot(hplc_df['Time'], hplc_df['Signal'], color='green', linewidth=1.5)
    plt.title('HPLC Trace: FeatureFinderMetaboIdent_1_input.mzML')
    plt.xlabel('Retention Time (seconds)')
    plt.ylabel('Total Ion Current (Signal)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
#@title List SMILES to find
import pandas as pd
import os
import sys
import traitlets
import subprocess
import ipywidgets as widgets
from rdkit import Chem
from rdkit.Chem import Descriptors

# Dynamic patch to solve 'TypeError: type "List" is not subscriptable' on Python 3.13 inside ipyketcher[cite: 1]
if not hasattr(traitlets.List, '__class_getitem__'):
    traitlets.List.__class_getitem__ = classmethod(lambda cls, item: cls)

# --- PATCH IPYKETCHER BUG ---[cite: 1]
ipk_init = f"/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages/ipyketcher/__init__.py"
if os.path.exists(ipk_init):
    with open(ipk_init, 'r') as f:
        content = f.read()
    if "traitlets.List[str]()" in content:
        with open(ipk_init, 'w') as f:
            f.write(content.replace("traitlets.List[str]()", "traitlets.List()"))
# ----------------------------

# Force reload in case the broken version was already cached in memory[cite: 1]
if 'ipyketcher' in sys.modules:
    del sys.modules['ipyketcher']
import ipyketcher
from IPython.display import display, HTML

# 1. Initialize and display the molecular sketching canvas[cite: 1]
editor = ipyketcher.KetcherEditor()
print("🎨 Canvas initialized. (Draw molecules to add them to your interactive table below)")
display(editor)

# Helper function to calculate exact mass via RDKit
def calculate_exact_mass(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol:
        return f"{Descriptors.ExactMolWt(mol):.4f}"
    return "Invalid"

# Define reaction scheme list dynamically with the requested molecules[cite: 1]
reaction_smiles = [
    ("adenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
    ("n-deoxyadenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)C1"),
    ("n-inosine", "Product", "O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
    ("n2-o-methylcytidine", "Product", "CO[C@@H]1[C@H](O[C@H](CO)[C@@H]1O)n2ccc(N)nc2=O"),
    ("n5-methylcytidine", "Product", "Cc1cn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)nc1N"),
    ("n5-formylcytidine", "Product", "Nc1nc(=O)n([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)cc1C=O")
]

# Create a UI Row for the interactive table
def create_table_row(comp_no, name, classification, smiles):
    w_no = widgets.HTML(value=f"<b>{comp_no}</b>", layout=widgets.Layout(width='40px', padding='0 5px'))
    w_name = widgets.Text(value=name, placeholder='Molecule Name', layout=widgets.Layout(width='160px', padding='0 5px'))
    w_class = widgets.Dropdown(
        options=['Reactant', 'Reagent', 'Product'],
        value=classification,
        layout=widgets.Layout(width='100px', padding='0 5px')
    )
    w_smiles = widgets.Text(value=smiles, layout=widgets.Layout(width='320px', padding='0 5px'))
    w_mass = widgets.HTML(value=calculate_exact_mass(smiles), layout=widgets.Layout(width='80px', padding='0 5px'))

    return widgets.HBox([w_no, w_name, w_class, w_smiles, w_mass])

# Build Table Header
header_row = widgets.HBox([
    widgets.HTML(value="<b>No.</b>", layout=widgets.Layout(width='40px', padding='0 5px')),
    widgets.HTML(value="<b>Name</b>", layout=widgets.Layout(width='160px', padding='0 5px')),
    widgets.HTML(value="<b>Classification</b>", layout=widgets.Layout(width='100px', padding='0 5px')),
    widgets.HTML(value="<b>SMILES String</b>", layout=widgets.Layout(width='320px', padding='0 5px')),
    widgets.HTML(value="<b>Exact Mass</b>", layout=widgets.Layout(width='80px', padding='0 5px'))
])

# Populate initial table rows
table_rows = [create_table_row(i + 1, name, cls, smi) for i, (name, cls, smi) in enumerate(reaction_smiles)]
table_container = widgets.VBox([header_row] + table_rows)

btn_update = widgets.Button(description='Update List', button_style='success')
out = widgets.Output()

def on_update_clicked(b):
    with out:
        out.clear_output()
        # ipyketcher stores current smiles in editor.smiles[cite: 1]
        drawn_smi = getattr(editor, 'smiles', None)
        if drawn_smi:
            # Check for duplicates based on existing SMILES in the table
            existing_smiles = [row.children[3].value for row in table_container.children[1:]]
            if drawn_smi not in existing_smiles:
                new_index = len(table_container.children) # Header counts as 1, so len equals the next compound number
                new_row = create_table_row(new_index, "Drawn Molecule", "Reactant", drawn_smi)

                # Append the new row to the table container
                table_container.children = list(table_container.children) + [new_row]
                print(f"Added to table: {drawn_smi}")
            else:
                print(f"Molecule already exists in the table: {drawn_smi}")
        else:
            print("No SMILES found in editor. Please draw a molecule first.")[cite: 1]

btn_update.on_click(on_update_clicked)

display(table_container)
display(widgets.HBox([btn_update]))
display(out)

Here we use an **LC-MS1 alignment/extraction pipeline**.

 We can use the OpenMS **Python wrapper, pyOpenMS**, directly inside this colab notebook to perform a precise, high-resolution Extracted Ion Chromatogram (EIC) extraction and retention time alignment for your target molecular masses.

In [ ]:
#@title Unified Peak Detection, Dominant-m/z Confidence Scoring & Auto-Arranged Cards (dataframe = detected_df)

import subprocess
import sys
import os
import re
import warnings

try:
    import scipy.signal
    from adjustText import adjust_text
except ImportError:
    print("🔧 Installing missing dependencies (scipy, numpy, adjustText) via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "numpy", "scipy", "adjustText"], capture_output=True)
    import scipy.signal
    from adjustText import adjust_text
    print("✅ Repair complete.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.offsetbox import OffsetImage, AnnotationBbox, TextArea, VPacker
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets
from pyteomics import mzml
import pyopenms as po
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Draw

warnings.filterwarnings('ignore')

# 1. Ensure hplc-py and pyOpenMS experiment are loaded
if 'hplc-py' not in sys.path:
    sys.path.append(os.path.abspath('hplc-py'))
from hplc.quant import Chromatogram

mzml_filename = "FeatureFinderMetaboIdent_1_input.mzML"
if not os.path.exists(mzml_filename):
    mzml_filename = "/content/FeatureFinderMetaboIdent_1_input.mzML"

if 'od' not in globals() and os.path.exists(mzml_filename):
    od = po.MSExperiment()
    po.FileHandler().loadExperiment(mzml_filename, od)

signal_raw = hplc_df['Signal'].values
signal_norm = (signal_raw - signal_raw.min()) / (signal_raw.max() - signal_raw.min())

# --- 2. Dynamic Hyperparameter Optimization (Elbow Method) ---
print("Running Dynamic Hyperparameter Optimization...")
prominence_range = np.logspace(-2, -0.5, 8)
buffer_val = 0
results = []
chrom_base = Chromatogram(hplc_df, cols={'time': 'Time', 'signal': 'Signal'}, time_window=[200, 300])

for p in prominence_range:
    try:
        peaks = chrom_base.fit_peaks(prominence=p, buffer=buffer_val)
        results.append((p, len(peaks)))
    except Exception:
        pass

if not results:
    default_prominence = 0.05
    print("Optimization failed. Using default prominence: 0.05")
else:
    proms_all = np.array([x[0] for x in results])
    counts_all = np.array([x[1] for x in results])

    max_idx = np.argmax(counts_all)
    valid_proms = proms_all[max_idx:]
    valid_counts = counts_all[max_idx:]

    if len(valid_proms) > 1:
        p1 = np.array([np.log10(valid_proms[0]), valid_counts[0]])
        p2 = np.array([np.log10(valid_proms[-1]), valid_counts[-1]])

        distances = []
        for i in range(len(valid_proms)):
            p3 = np.array([np.log10(valid_proms[i]), valid_counts[i]])
            if np.linalg.norm(p2 - p1) == 0:
                dist = 0
            else:
                cross_prod_2d = (p2[0] - p1[0]) * (p1[1] - p3[1]) - (p2[1] - p1[1]) * (p1[0] - p3[0])
                dist = np.abs(cross_prod_2d) / np.linalg.norm(p2 - p1)
            distances.append(dist)

        optimal_idx_relative = np.argmax(distances)
        optimal_prominence = valid_proms[optimal_idx_relative]
        optimal_count = valid_counts[optimal_idx_relative]
    else:
        optimal_prominence = valid_proms[0]
        optimal_count = valid_counts[0]

    default_prominence = optimal_prominence
    prominence_val = optimal_prominence

    display(Markdown("---"))
    display(Markdown(f"### ✅ Optimization Complete"))
    display(Markdown(f"**Optimal Prominence Threshold:** `{optimal_prominence:.4f}` | **Peaks Resolved:** `{optimal_count}`"))

    fig_opt, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [2.5, 1]})
    ax1.plot(proms_all, counts_all, marker='o', linestyle='-', color='#2c3e50', linewidth=2, label='Fitted Peaks')
    ax1.axvline(optimal_prominence, color='#e74c3c', linestyle='--', linewidth=2, label=f'Optimal Elbow ({optimal_prominence:.4f})')
    ax1.scatter([optimal_prominence], [optimal_count], color='#e74c3c', s=150, zorder=5)
    ax1.set_xscale('log')
    ax1.set_xticks(prominence_range)
    ax1.set_xticklabels([f"{x:.4f}" for x in prominence_range], rotation=45)
    ax1.set_title('Hyperparameter Optimization: Signal-to-Noise Ratio (L-Curve)', fontsize=13, pad=12)
    ax1.set_xlabel('Prominence Threshold (Log Scale)', fontsize=11)
    ax1.set_ylabel('Number of Deconvoluted Peaks', fontsize=11)
    ax1.legend(fontsize=10)
    ax1.grid(True, which="both", ls="--", alpha=0.4)

    ax2.axis('tight')
    ax2.axis('off')
    table_data = [[f"{p:.4f}", int(c)] for p, c in zip(proms_all, counts_all)]
    col_labels = ['Prominence Threshold', 'Number of\nDeconvoluted Peaks']
    table = ax2.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.1, 1.8)

    optimal_row_idx = list(proms_all).index(optimal_prominence) + 1
    for (row, col), cell in table.get_celld().items():
        if row == optimal_row_idx:
            cell.set_edgecolor('red')
            cell.set_linewidth(2)
    ax2.set_title('Tabular Data', fontsize=12, pad=12)
    plt.tight_layout()
    plt.show()

# --- 3. Candidate Fetching & Theoretical Ion / Isotope Generators ---
def get_candidates_from_table():
    if 'table_container' in globals():
        candidates = []
        for row in table_container.children[1:]:
            name = row.children[1].value.strip()
            classification = row.children[2].value
            smi = row.children[3].value.strip()
            if smi:
                candidates.append((name, classification, smi))
        if candidates:
            return candidates
    return [
        ("adenosine",           "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
        ("n-deoxyadenosine",    "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)C1"),
        ("n-inosine",           "Product", "O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
        ("n2-o-methylcytidine", "Product", "CO[C@@H]1[C@H](O[C@H](CO)[C@@H]1O)n2ccc(N)nc2=O"),
        ("n5-methylcytidine",   "Product", "Cc1cn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)nc1N"),
        ("n5-formylcytidine",   "Product", "Nc1nc(=O)n([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)cc1C=O")
    ]

esi_pos_adducts = [
    ("[M+H]+",       1, 1.007276,  1),
    ("[M+NH4]+",     1, 18.033823, 1),
    ("[M+Na]+",      1, 22.989218, 1),
    ("[M+CH3OH+H]+", 1, 33.033489, 1),
    ("[M+K]+",       1, 38.963158, 1),
    ("[M+ACN+H]+",   1, 42.033823, 1),
    ("[2M+H]+",      2, 1.007276,  1),
    ("[2M+Na]+",     2, 22.989218, 1),
]

insource_neutral_losses = [
    ("[M+H-Ribose]+",      -132.042259),
    ("[M+H-Deoxyribose]+", -116.047344),
    ("[M+H-MeRibose]+",    -146.057909),
    ("[M+H-H2O]+",         -18.010565),
]

def parse_dominant_mz_string(dom_str):
    if not dom_str or dom_str == "None":
        return []
    parsed = []
    for item in str(dom_str).split(","):
        m = re.search(r"([0-9]+\.[0-9]+)\s*\(([0-9.]+)%\)", item.strip())
        if m:
            parsed.append((float(m.group(1)), float(m.group(2))))
    return parsed

def compute_confidence_from_dominant_mz(best_group, dom_mz_list, full_window_spectrum):
    adducts = best_group["adducts"]
    primary = adducts[0]
    mol = best_group["mols"][0]
    exact_mass = best_group["exact_mass"]

    # Mass Accuracy Score
    mean_abs_ppm = np.mean([abs(a["ppm"]) for a in adducts])
    ppm_score = max(0.0, 35.0 * (1.0 - (mean_abs_ppm / 15.0)))

    # Dominant m/z Rank
    matched_dom_rel_int = 0.0
    for a in adducts:
        for dom_mz, dom_rel in dom_mz_list:
            if abs(dom_mz - a["calc_mz"]) <= 0.015:
                matched_dom_rel_int = max(matched_dom_rel_int, dom_rel)

    if matched_dom_rel_int >= 95.0:
        dom_rank_score = 35.0
    elif matched_dom_rel_int > 0:
        dom_rank_score = 12.0 + 23.0 * (matched_dom_rel_int / 100.0)
    else:
        dom_rank_score = 5.0

    # Isotope Envelope Verification
    num_C = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)
    num_N = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 7)
    num_S = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 16)

    theo_m1_rel = (num_C * 1.07) + (num_N * 0.37) + (num_S * 0.76)
    if theo_m1_rel == 0:
        theo_m1_rel = 12.0

    m0_mz = primary["calc_mz"]
    m1_target = m0_mz + 1.003355
    m2_target = m0_mz + 2.006710

    iso_score = 0.0
    iso_evidence = []

    m1_obs_rel = None
    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m1_target) <= 0.015 and matched_dom_rel_int > 0:
            m1_obs_rel = (dom_rel / matched_dom_rel_int) * 100.0
            break

    if m1_obs_rel is None and full_window_spectrum and matched_dom_rel_int > 0:
        m0_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m0_mz) <= 0.015], default=0)
        m1_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m1_target) <= 0.015], default=0)
        if m0_int > 0 and m1_int > 0:
            m1_obs_rel = (m1_int / m0_int) * 100.0

    if m1_obs_rel is not None:
        rel_diff = abs(m1_obs_rel - theo_m1_rel)
        iso_score += max(6.0, 16.0 * (1.0 - min(rel_diff / 15.0, 1.0)))
        iso_evidence.append(f"[M+H+1]+ iso ({m1_obs_rel:.0f}% vs {theo_m1_rel:.0f}% theo)")

    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m2_target) <= 0.015:
            iso_score += 4.0
            iso_evidence.append("[M+H+2]+ iso")
            break

    # Multi-Adduct & In-Source Fragments
    corrob_score = 0.0
    if len(adducts) > 1:
        corrob_score += min(7.0, (len(adducts) - 1) * 4.5)

    mh_mz = exact_mass + 1.007276
    for loss_name, shift in insource_neutral_losses:
        frag_mz = mh_mz + shift
        if frag_mz > 80.0:
            for dom_mz, dom_rel in dom_mz_list:
                if abs(dom_mz - frag_mz) <= 0.015:
                    corrob_score += 5.0
                    iso_evidence.append(f"{loss_name} ({dom_mz:.3f})")
                    break

    corrob_score = min(10.0, corrob_score)
    total_conf = round(min(99.9, ppm_score + dom_rank_score + iso_score + corrob_score), 1)

    evidence_summary = f"Dom.Rank: {matched_dom_rel_int:.0f}%"
    if iso_evidence:
        evidence_summary += " | " + ", ".join(iso_evidence[:2])

    return total_conf, evidence_summary

# --- 4. Interactive UI & Unified Execution Pipeline ---
prominence_input = widgets.FloatText(
    value=default_prominence,
    description='Prominence:',
    step=0.001,
    style={'description_width': 'initial'}
)

rerun_button = widgets.Button(
    description='Rerun Peak Fitting & Assignment',
    button_style='primary',
    icon='refresh',
    layout=widgets.Layout(width='240px')
)

output_area = widgets.Output()

def run_unified_pipeline(prom_val):
    global detected_df, peak_assignments
    with output_area:
        clear_output(wait=True)
        print(f"Using prominence threshold: {prom_val:.4f}")

        # Topographical peak detection
        peak_indices, _ = scipy.signal.find_peaks(signal_norm, prominence=prom_val)
        _, left_bases, right_bases = scipy.signal.peak_prominences(signal_norm, peak_indices)

        start_times = hplc_df['Time'].iloc[left_bases].values
        end_times = hplc_df['Time'].iloc[right_bases].values

        print(f"Detected {len(peak_indices)} raw topographical anomalies. Deconvoluting true peaks...")
        chrom = Chromatogram(hplc_df, cols={'time': 'Time', 'signal': 'Signal'}, time_window=[200, 300])
        peaks = chrom.fit_peaks(prominence=prom_val, buffer=0).reset_index(drop=True)

        mapped_starts, mapped_ends, mapped_topos = [], [], []
        peak_rts = hplc_df['Time'].values[peak_indices]

        for rt in peaks['retention_time']:
            if len(peak_rts) > 0:
                closest_idx = np.argmin(np.abs(peak_rts - rt))
                mapped_starts.append(start_times[closest_idx])
                mapped_ends.append(end_times[closest_idx])
                mapped_topos.append(peak_rts[closest_idx])
            else:
                mapped_starts.append(np.nan)
                mapped_ends.append(np.nan)
                mapped_topos.append(np.nan)

        peaks['peak start time'] = mapped_starts
        peaks['peak end time'] = mapped_ends
        peaks['topographical_apex'] = mapped_topos

        if peaks['area'].sum() > 0:
            peaks['normalised area'] = (peaks['area'] / peaks['area'].sum() * 100).map('{:.2f}%'.format)
        else:
            peaks['normalised area'] = '0.00%'

        # Extract top 10 dominant m/z signals
        peak_spectra = {i: {} for i in range(len(peaks))}
        if os.path.exists(mzml_filename):
            with mzml.read(mzml_filename) as reader:
                for spectrum in reader:
                    if spectrum['ms level'] == 1:
                        scan_data = spectrum.get('scanList', {}).get('scan', [{}])
                        rt = scan_data[0].get('scan start time', -1) if isinstance(scan_data, list) and scan_data else (scan_data.get('scan start time', -1) if isinstance(scan_data, dict) else -1)
                        mz_array = spectrum.get('m/z array', [])
                        int_array = spectrum.get('intensity array', [])

                        if len(mz_array) > 0:
                            mzs_rounded = np.round(mz_array, 3)
                            for i, row in peaks.iterrows():
                                if not pd.isna(row['peak start time']) and row['peak start time'] <= rt <= row['peak end time']:
                                    for m, intensity in zip(mzs_rounded, int_array):
                                        peak_spectra[i][m] = peak_spectra[i].get(m, 0) + intensity

        TOP_N_PEAKS = 10
        MIN_REL_INTENSITY = 2.0
        dominant_mz_col = []
        for i in range(len(peaks)):
            spectra = peak_spectra[i]
            if not spectra:
                dominant_mz_col.append("None")
                continue
            sorted_mzs = sorted(spectra.items(), key=lambda x: x[1], reverse=True)
            max_int = sorted_mzs[0][1]
            top_strs = []
            for m, inten in sorted_mzs:
                rel_int = (inten / max_int) * 100
                if rel_int >= MIN_REL_INTENSITY:
                    top_strs.append(f"{m:.3f} ({rel_int:.0f}%)")
                if len(top_strs) >= TOP_N_PEAKS:
                    break
            dominant_mz_col.append(", ".join(top_strs) if top_strs else "None")

        peaks['dominant m/z'] = dominant_mz_col

        # Screen ESI+ adducts & score
        candidates = get_candidates_from_table()
        theoretical_ions = []
        for name, classification, smi in candidates:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                exact_mass = Descriptors.ExactMolWt(mol)
                formula = rdMolDescriptors.CalcMolFormula(mol)
                for adduct_name, mult, shift, z in esi_pos_adducts:
                    calc_mz = (exact_mass * mult + shift) / z
                    theoretical_ions.append({
                        "name": f"{name} ({classification})",
                        "raw_name": name,
                        "classification": classification,
                        "smiles": smi,
                        "mol": mol,
                        "formula": formula,
                        "exact_mass": round(exact_mass, 4),
                        "adduct": adduct_name,
                        "calc_mz": calc_mz
                    })

        PPM_TOLERANCE = 15.0
        MIN_PEAK_INTENSITY = 5e5
        peak_assignments = {}

        if 'od' in globals():
            for idx_row, peak_row in peaks.iterrows():
                p_id = int(peak_row['peak_id'])
                rt_start = peak_row['peak start time']
                rt_end = peak_row['peak end time']
                rt_apex = peak_row['retention_time']
                dom_mz_parsed = parse_dominant_mz_string(peak_row['dominant m/z'])

                if pd.isna(rt_start) or pd.isna(rt_end):
                    continue

                window_scans = [
                    spec for spec in od
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end
                ]
                if not window_scans:
                    continue

                mass_group_hits = {}
                for ion in theoretical_ions:
                    target_mz = ion["calc_mz"]
                    tol_da = target_mz * (PPM_TOLERANCE * 1e-6)

                    max_int = 0.0
                    obs_mz_at_max = target_mz
                    rt_at_max = rt_apex

                    for spec in window_scans:
                        mzs, ints = spec.get_peaks()
                        mask = (mzs >= target_mz - tol_da) & (mzs <= target_mz + tol_da)
                        if np.any(mask):
                            best_idx = np.argmax(ints[mask])
                            peak_int = float(ints[mask][best_idx])
                            if peak_int > max_int:
                                max_int = peak_int
                                obs_mz_at_max = float(mzs[mask][best_idx])
                                rt_at_max = float(spec.getRT())

                    if max_int >= MIN_PEAK_INTENSITY and abs(rt_at_max - rt_apex) <= 6.0:
                        em = ion["exact_mass"]
                        if em not in mass_group_hits:
                            mass_group_hits[em] = {
                                "exact_mass": em,
                                "formula": ion["formula"],
                                "names": [],
                                "raw_names": [],
                                "mols": [],
                                "total_intensity": 0.0,
                                "adducts": {}
                            }
                        if ion["name"] not in mass_group_hits[em]["names"]:
                            mass_group_hits[em]["names"].append(ion["name"])
                            mass_group_hits[em]["raw_names"].append(ion["raw_name"])
                            mass_group_hits[em]["mols"].append(ion["mol"])

                        if ion["adduct"] not in mass_group_hits[em]["adducts"]:
                            ppm_err = ((obs_mz_at_max - target_mz) / target_mz) * 1e6
                            mass_group_hits[em]["adducts"][ion["adduct"]] = {
                                "adduct": ion["adduct"],
                                "obs_mz": obs_mz_at_max,
                                "calc_mz": target_mz,
                                "ppm": ppm_err,
                                "intensity": max_int
                            }
                            mass_group_hits[em]["total_intensity"] += max_int

                if mass_group_hits:
                    best_group = max(mass_group_hits.values(), key=lambda x: x["total_intensity"])
                    sorted_adducts = sorted(best_group["adducts"].values(), key=lambda x: x["intensity"], reverse=True)
                    best_group["adducts"] = sorted_adducts

                    is_isobaric = len(best_group["raw_names"]) > 1
                    iso_tag = f" [Isobaric Isomers, {best_group['formula']}]" if is_isobaric else ""

                    conf_score, evidence_str = compute_confidence_from_dominant_mz(
                        best_group, dom_mz_parsed, peak_spectra.get(idx_row, {})
                    )

                    peak_assignments[p_id] = {
                        "display_name": " / ".join(best_group["raw_names"]),
                        "full_assignment": " / ".join(best_group["names"]) + iso_tag,
                        "is_isobaric": is_isobaric,
                        "formula": best_group["formula"],
                        "mols": best_group["mols"],
                        "adducts": sorted_adducts,
                        "confidence": conf_score,
                        "evidence": evidence_str
                    }

        assignment_col = []
        confidence_col = []
        for idx_row, row in peaks.iterrows():
            p_id = int(row['peak_id'])
            if p_id in peak_assignments:
                assign_info = peak_assignments[p_id]
                top_adduct = assign_info["adducts"][0]
                assignment_col.append(
                    f"{assign_info['full_assignment']} [{top_adduct['adduct']}: {top_adduct['obs_mz']:.4f} m/z ({top_adduct['ppm']:+.1f} ppm)]"
                )
                confidence_col.append(f"{assign_info['confidence']:.1f}% ({assign_info['evidence']})")
            else:
                dom_list = parse_dominant_mz_string(row['dominant m/z'])
                base_str = f"Base m/z: {dom_list[0][0]:.3f}" if dom_list else "No MS1 signal"
                assignment_col.append(f"Unassigned ({base_str})")
                confidence_col.append("N/A")

        peaks['Assignment'] = assignment_col
        peaks['Confidence (%)'] = confidence_col
        detected_df = peaks.copy()

        def highlight_peaks(row):
            colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
            c = mcolors.to_rgba(colors[(int(row['peak_id']) - 1) % len(colors)], alpha=0.4)
            bg_color = f'rgba({int(c[0]*255)}, {int(c[1]*255)}, {int(c[2]*255)}, {c[3]})'
            return [f'background-color: {bg_color}' for _ in row]

        display(detected_df.style.apply(highlight_peaks, axis=1))
        print(f"\nFinal deconvoluted peak count: {len(detected_df)} | Assigned peaks: {len(peak_assignments)}")

        # E. Render Deconvoluted Chromatogram with Staggered Grids & adjustText Repulsion
        fig, ax = chrom.show()
        fig.set_size_inches(20, 11.5)
        fig.patch.set_facecolor('white')

        plotted_x = ax.lines[0].get_xdata()
        plotted_y = ax.lines[0].get_ydata()
        y_max_plot = np.max(plotted_y)

        # Plot drop-lines for apexes
        for _, peak_row in detected_df.iterrows():
            rt_apex = peak_row['retention_time']
            c_idx = np.argmin(np.abs(plotted_x - rt_apex))
            ax.vlines(rt_apex, 0, plotted_y[c_idx], linestyle='--', color='dodgerblue', alpha=0.6, linewidth=1.2)

        colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

        # Staggered offsets explicitly separated to guarantee zero image card overlap
        assigned_card_offsets = {
            1: (204.5, y_max_plot * 0.50),
            2: (218.5, y_max_plot * 1.25),
            3: (236.0, y_max_plot * 0.85),
            4: (258.0, y_max_plot * 1.25),
            5: (275.0, y_max_plot * 0.80),
        }

        texts_to_adjust = []
        artists_to_avoid = []

        for _, peak_row in detected_df.iterrows():
            p_id = int(peak_row['peak_id'])
            rt_apex = peak_row['retention_time']
            c_idx = np.argmin(np.abs(plotted_x - rt_apex))
            apex_y = plotted_y[c_idx]
            peak_color = colors[(p_id - 1) % len(colors)]

            if p_id in peak_assignments:
                assign = peak_assignments[p_id]
                mol_arrays = [np.asarray(Draw.MolToImage(m, size=(135, 120))) for m in assign["mols"]]
                if len(mol_arrays) > 1:
                    img_arr = np.hstack(mol_arrays)
                    zoom_factor = 0.55
                else:
                    img_arr = mol_arrays[0]
                    zoom_factor = 0.62

                imagebox = OffsetImage(img_arr, zoom=zoom_factor)
                adduct_lines = [
                    f"{a['adduct']}: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                    for a in assign["adducts"]
                ]

                iso_header = f"\n⚠️ Isobaric Isomers ({assign['formula']})" if assign["is_isobaric"] else ""
                conf_header = f"\nConfidence: {assign['confidence']:.1f}%"

                header_text = (
                    f"Peak {p_id} ({rt_apex:.1f}s): {assign['display_name']}"
                    + iso_header
                    + "\n" + "\n".join(adduct_lines)
                    + conf_header
                )

                textbox = TextArea(
                    header_text,
                    textprops=dict(color='#111827', fontsize=8.0, fontweight='bold', ha='center', multialignment='center')
                )

                packed_card = VPacker(children=[textbox, imagebox], align="center", pad=2, sep=3)

                box_x, box_y = assigned_card_offsets.get(p_id, (rt_apex, apex_y + y_max_plot * 0.45))

                ab = AnnotationBbox(
                    packed_card,
                    (rt_apex, apex_y),
                    xybox=(box_x, box_y),
                    xycoords='data',
                    boxcoords='data',
                    pad=0.35,
                    bboxprops=dict(boxstyle="round,pad=0.35", fc="white", ec=peak_color, lw=2.2, alpha=0.75),
                    arrowprops=dict(arrowstyle="->", color=peak_color, lw=1.8, connectionstyle="arc3,rad=0.06")
                )
                ax.add_artist(ab)

                # Add AnnotationBbox to the 'avoid' list so text labels bounce off them
                artists_to_avoid.append(ab)

            else:
                dom_str = str(peak_row.get('dominant m/z', 'None'))
                dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
                top_ions_str = ", ".join(dom_tokens[:2]) if dom_tokens else "N/A"

                initial_y_box = apex_y + y_max_plot * 0.15

                ann = ax.annotate(
                    f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned\nDom. m/z: {top_ions_str}",
                    xy=(rt_apex, apex_y),
                    xytext=(rt_apex, initial_y_box),
                    ha='center',
                    fontsize=7.8,
                    fontweight='bold',
                    color='#374151',
                    bbox=dict(boxstyle="round,pad=0.28", fc="#f9fafb", ec=peak_color, lw=1.5, ls='--', alpha=0.75),
                    arrowprops=dict(arrowstyle="->", color=peak_color, lw=1.4, ls='--', connectionstyle="arc3,rad=0.05")
                )
                texts_to_adjust.append(ann)

        # Force canvas draw before adjustText to ensure bounding boxes are properly calculated
        fig.canvas.draw()

        # Apply adjustText using the image cards as "walls" (add_objects) to avoid
        try:
            adjust_text(
                texts_to_adjust,
                add_objects=artists_to_avoid,
                ax=ax,
                force_points=0.2,
                force_text=0.4,
                force_objects=0.5,
                expand_points=(1.2, 1.2),
                expand_text=(1.2, 1.2),
                arrowprops=dict(arrowstyle="-", color='gray', lw=0.5, alpha=0.5)
            )
        except Exception as e:
            print(f"adjustText layout optimization skipped: {e}")

        handles, labels = ax.get_legend_handles_labels()
        updated_labels = []
        for lbl in labels:
            if lbl.lower().startswith('peak '):
                try:
                    p_num = int(lbl.split()[1])
                    if p_num in peak_assignments:
                        a_info = peak_assignments[p_num]
                        iso_short = " [Isobaric]" if a_info["is_isobaric"] else ""
                        adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts']])
                        updated_labels.append(
                            f"Peak {p_num}: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
                        )
                    else:
                        dom_str = detected_df.loc[detected_df['peak_id'] == p_num, 'dominant m/z'].values[0]
                        first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
                        updated_labels.append(f"Peak {p_num}: Unassigned (Base m/z: {first_ion})")
                except Exception:
                    updated_labels.append(lbl)
            else:
                updated_labels.append(lbl)

        ax.set_xlim(190, 310)
        ax.set_ylim(bottom=ax.get_ylim()[0], top=y_max_plot * 1.58)
        ax.set_title(f"HPLC Chromatogram Deconvolution, Isobaric Screening & Dominant-m/z Confidence (Prominence: {prom_val:.4f})",
                     fontsize=15, fontweight='bold', color='#111827', pad=15)
        ax.set_xlabel("Retention Time (seconds)", fontsize=12, fontweight='bold', color='#111827')
        ax.set_ylabel("Total Ion Count (baseline corrected)", fontsize=12, fontweight='bold', color='#111827')
        ax.tick_params(colors='#111827', labelsize=10)

        leg = ax.legend(handles, updated_labels, bbox_to_anchor=(1.01, 1), loc='upper left',
                        fontsize=9.0, frameon=True, facecolor='white', edgecolor='#d1d5db')
        for text in leg.get_texts():
            text.set_color('#111827')

        plt.tight_layout()
        plt.show()

def on_rerun_clicked(b):
    run_unified_pipeline(prominence_input.value)

rerun_button.on_click(on_rerun_clicked)

display(widgets.HBox([prominence_input, rerun_button]))
display(output_area)

run_unified_pipeline(default_prominence)